<a href="https://colab.research.google.com/github/mayaraisantos/projeto-etl-iqvia-clamed/blob/main/coleta_e_tratamento_ensaios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import os
from datetime import datetime

print("1. Iniciando consumo da API do ClinicalTrials.gov...")

# URL da API REST pública do ClinicalTrials.gov (v2)
# Buscando ensaios clínicos sobre estudos farmacêuticos/medicamentos
url = "https://clinicaltrials.gov/api/v2/studies?query.term=pharmaceutical&pageSize=50"

headers = {
    'accept': 'application/json'
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    studies = data.get('studies', [])
    print(f"Sucesso! {len(studies)} estudos encontrados.")
else:
    print(f"Erro ao acessar API: {response.status_code}")
    studies = []

# Extração e Estruturação dos Dados (Tratamento)
lista_estudos = []

for study in studies:
    protocol = study.get('protocolSection', {})

    # Identificação
    id_info = protocol.get('identificationModule', {})
    nct_id = id_info.get('nctId', 'N/A')
    title = id_info.get('briefTitle', 'N/A')

    # Status e Datas
    status_module = protocol.get('statusModule', {})
    overall_status = status_module.get('overallStatus', 'N/A')
    start_date_dict = status_module.get('startDateStruct', {})
    start_date = start_date_dict.get('date', 'N/A')

    # Desenho do Estudo / Fase
    design_module = protocol.get('designModule', {})
    phases = design_module.get('phases', ['N/A'])
    phase = phases[0] if phases else 'N/A'

    # Patrocinador / Organização
    sponsor_module = protocol.get('sponsorCollaboratorsModule', {})
    lead_sponsor = sponsor_module.get('leadSponsor', {}).get('name', 'N/A')

    # Condição / Doença
    conditions_module = protocol.get('conditionsModule', {})
    conditions = ", ".join(conditions_module.get('conditions', ['N/A']))

    lista_estudos.append({
        'NCT_ID': nct_id,
        'Titulo_Estudo': title,
        'Status': overall_status,
        'Fase': phase,
        'Patrocinador': lead_sponsor,
        'Condicao_Doenca': conditions,
        'Data_Inicio': start_date,
        'Data_Coleta': datetime.now().strftime('%Y-%m-%d')
    })

# Transformação em DataFrame (Pandas)
df = pd.DataFrame(lista_estudos)

# Tratamento e Limpeza de Dados
df.drop_duplicates(subset=['NCT_ID'], inplace=True)
df.fillna('Não informado', inplace=True)

# RPA Simulado / Organização de Arquivos
print("2. Executando etapa de RPA (Organização local do arquivo)...")
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "base_ensaios_clinicos.csv")
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f"Sucesso! Arquivo salvo em: {csv_path}")
print("\nPrimeiras 5 linhas da base estruturada:")
print(df.head())

1. Iniciando consumo da API do ClinicalTrials.gov...
Sucesso! 50 estudos encontrados.
2. Executando etapa de RPA (Organização local do arquivo)...
Sucesso! Arquivo salvo em: data/base_ensaios_clinicos.csv

Primeiras 5 linhas da base estruturada:
        NCT_ID                                      Titulo_Estudo  \
0  NCT03741530  Glibenclamide Advantage in Treating Edema Afte...   
1  NCT07036328  Transcranial Magnetic Stimulation to Slow Down...   
2  NCT00874328  A Study of TS-1 Plus Irinotecan and Cisplatin ...   
3  NCT07629765  Exploring How Osteopathic Manipulative Treatme...   
4  NCT05163665  Cost-effective Analysis of Two Approximation D...   

               Status    Fase  \
0           COMPLETED      NA   
1          RECRUITING      NA   
2             UNKNOWN  PHASE1   
3  NOT_YET_RECRUITING      NA   
4           COMPLETED      NA   

                                        Patrocinador  \
0                                    Xijing Hospital   
1                           